# Product Recommendation System (Market Basket Analysis)


### 1. Import + Loading Data

In [15]:
import pandas as pd

df = pd.read_excel("C:/Users/Lenovo/Online_Retail.xlsx", engine='openpyxl')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### 2. Data Cleaning

In [17]:
df = df.dropna(subset=['CustomerID'])
df = df[df['Quantity'] > 0]

### 3. Create Basket Matrix

In [20]:
basket = df.groupby(['InvoiceNo', 'Description'])['Quantity'] \
           .sum().unstack().fillna(0)

### 4. Convert to Binary

In [26]:
basket = basket.applymap(lambda x: 1 if x > 0 else 0)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_2092\3498954818.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket = basket.applymap(lambda x: 1 if x > 0 else 0)


### 5. Apply Apriori

In [27]:
from mlxtend.frequent_patterns import apriori, association_rules
frequent_items = apriori(basket, min_support=0.02, use_colnames=True)
rules = association_rules(frequent_items, metric='lift', min_threshold=1)

C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [ ]:
### 6. filter Strong Rules 

In [28]:
strong_rules = rules[
    (rules['confidence'] > 0.5) &
    (rules['lift'] > 1.5)
]
strong_rules.sort_values(by='lift', ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
74,(PINK REGENCY TEACUP AND SAUCER),"(GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...",0.029996,0.029186,0.021040,0.701439,24.033032,1.0,0.020165,3.251641,0.988027,0.551627,0.692463,0.711163
71,"(GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...",(PINK REGENCY TEACUP AND SAUCER),0.029186,0.029996,0.021040,0.720887,24.033032,1.0,0.020165,3.475313,0.987204,0.551627,0.712256,0.711163
73,(GREEN REGENCY TEACUP AND SAUCER),"(PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY...",0.037279,0.023522,0.021040,0.564399,23.994742,1.0,0.020163,2.241683,0.995433,0.529172,0.553907,0.729447
72,"(PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY...",(GREEN REGENCY TEACUP AND SAUCER),0.023522,0.037279,0.021040,0.894495,23.994742,1.0,0.020163,9.124923,0.981409,0.529172,0.890410,0.729447
8,(GREEN REGENCY TEACUP AND SAUCER),(PINK REGENCY TEACUP AND SAUCER),0.037279,0.029996,0.024817,0.665702,22.193256,1.0,0.023698,2.901615,0.991919,0.584498,0.655364,0.746520
9,(PINK REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.029996,0.037279,0.024817,0.827338,22.193256,1.0,0.023698,5.575760,0.984471,0.584498,0.820652,0.746520
70,"(GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY...",(ROSES REGENCY TEACUP AND SAUCER ),0.024817,0.042242,0.021040,0.847826,20.070631,1.0,0.019992,6.293837,0.974356,0.457210,0.841114,0.672955
63,(ROSES REGENCY TEACUP AND SAUCER ),(PINK REGENCY TEACUP AND SAUCER),0.042242,0.029996,0.023522,0.556833,18.563760,1.0,0.022255,2.188799,0.987861,0.482835,0.543129,0.670503
62,(PINK REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER ),0.029996,0.042242,0.023522,0.784173,18.563760,1.0,0.022255,4.437611,0.975389,0.482835,0.774654,0.670503
12,(GREEN REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER ),0.037279,0.042242,0.029186,0.782923,18.534184,1.0,0.027612,4.412071,0.982679,0.579850,0.773349,0.736928


In [7]:
### 7. Recommendation Function

In [29]:
def recommend_products(product_name):
    recs = strong_rules[
        strong_rules['antecedents'].apply(lambda x: product_name in x)
    ]
    recs = recs.sort_values(by='lift', ascending=False)
    return recs[['antecedents', 'consequents']].head(5)

In [8]:
###8. Test Recommendation 

In [30]:
recommend_products("WHITE HANGING HEART T-LIGHT HOLDER")

,antecedents,consequents


In [31]:
df['Description'].value_counts().head(10) # Finding top selling product

Description
WHITE HANGING HEART T-LIGHT HOLDER    2028
REGENCY CAKESTAND 3 TIER              1724
JUMBO BAG RED RETROSPOT               1618
ASSORTED COLOUR BIRD ORNAMENT         1408
PARTY BUNTING                         1397
LUNCH BAG RED RETROSPOT               1316
SET OF 3 CAKE TINS PANTRY DESIGN      1159
LUNCH BAG  BLACK SKULL.               1105
POSTAGE                               1099
PACK OF 72 RETROSPOT CAKE CASES       1068
Name: count, dtype: int64

In [32]:
final_output = strong_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
final_output['antecedents'] = final_output['antecedents'].apply(lambda x: ', '.join(list(x)))
final_output['consequents'] = final_output['consequents'].apply(lambda x: ', '.join(list(x)))

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_2092\1099844939.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_output['antecedents'] = final_output['antecedents'].apply(lambda x: ', '.join(list(x)))
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_2092\1099844939.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_output['consequents'] = final_output['consequents'].apply(lambda x: ', '.join(list(x)))


In [33]:
import os
os.makedirs("outputs", exist_ok=True)
final_output.to_csv("outputs/final_recommendations.csv", index=False)